# R13 transport - structure: H134 neighborhood Wasserstein, H135 assignment-constrained resolution, H136 Gromov-Wasserstein twins

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R13 optimal-transport round, CPU-now batch <br>
**Graph**: rebuilt CPAP graph (neo4j2, read-only) <br>

Three optimal-transport tests over graph structure - the neighborhood, the assignment polytope, the
metric-space alignment. All use the stored Titan embeddings; no token embedder needed.

**H134** - neighborhood Wasserstein generalizes H62's neighbor-set Jaccard: an entity is a
distribution over its neighbors' embeddings, so duplicates whose neighbor sets differ in surface
form but agree in meaning still transport cheaply. Bar: >= 0.05 AUC over Jaccard AND signal on
zero-overlap pairs. Refuter: median degree 1 starves the signal.

**H135** - the transport polytope forbids the chain. False SAME_AS closures were greedy pairwise
merges then transitive union - nothing enforced global consistency. One-to-one assignment (low-temp
Sinkhorn) makes "A~B and A~C with B unlike C" expensive by construction. Bar: zero false closures
survive + legit multi-doc identity (the P10 family) regression-free. Refuter: legit multi-facet
entities NEED many-to-one.

**H136** - Gromov-Wasserstein aligns two metric spaces without a shared embedding: it matches
neighborhoods by internal distance pattern. Duplicates whose text diverged completely (code vs full
name) may be structural twins. Bar: >= 3 unique recoveries at precision >= 0.5 that both cosine and
string rank below threshold. Refuter: degree-1 neighborhoods make GW degenerate.

## Outputs
- H134: neighborhood-Wasserstein vs Jaccard AUC, coverage, zero-overlap subset
- H135: false-edge vs legit-edge survival under per-cluster Sinkhorn assignment + P10 case
- H136: GW-flagged head recoveries, subgraph-degeneracy coverage
- report JSON to ../reports/transport-r13-structure-<UTC>.json

In [1]:
# Imports (CPU only - Titan embeddings from the graph, POT for transport)
# stdlib
import datetime, json, glob, re
from pathlib import Path
from collections import Counter, defaultdict

# third party
import numpy as np
import ot  # POT - EMD, Sinkhorn, Gromov-Wasserstein
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from rich import print as rprint
from neo4j import GraphDatabase
try:
    import Levenshtein
    def jw(a, b):
        return Levenshtein.jaro_winkler(a.casefold(), b.casefold())
except Exception:
    def jw(a, b):
        return 0.0

np.random.seed(42)

In [2]:
# Configuration
NEO4J_URI = "bolt://user-konrad.jelen-kgf-neo4j2:7687"  # read-only graph
STRUCT_EXCLUDE = ["ABOUT", "MENTIONED_IN", "SIMILAR_TO", "SAME_AS"]  # non-semantic edges
DUP_CLASSES = ("variance", "resolver_miss", "samename")
SIB_CLASSES = ("sibling",)
SINKHORN_ASSIGN_REG = 0.01     # H135 low temperature -> near-hard assignment
BAR_H134_GAIN = 0.05           # neighborhood-W AUC over Jaccard AUC
BAR_H135_LEGIT = 0.90          # legit-edge survival (regression-free)
BAR_H136_RECOV = 3            # unique GW recoveries
BAR_H136_PREC = 0.50           # precision in the flagged head
COS_MISS = 0.85                # H136: "cosine ranks below threshold"
JW_MISS = 0.70                 # H136: "string ranks below threshold"

_l = "─" * 46
rprint(f"[bold cyan]Configuration[/bold cyan]\n[dim]{_l}[/dim]")
rprint(f"  Graph            : [cyan]{NEO4J_URI}[/cyan] [dim](read-only)[/dim]")
rprint(f"  Semantic nbhd    : all rels except [yellow]{STRUCT_EXCLUDE}[/yellow]")
rprint(f"  Bars             : H134 gain >= [yellow]{BAR_H134_GAIN}[/yellow]  H135 legit >= [yellow]{BAR_H135_LEGIT}[/yellow]  H136 >= [yellow]{BAR_H136_RECOV}[/yellow] recov @prec [yellow]{BAR_H136_PREC}[/yellow]")

Configuration
──────────────────────────────────────────────

Graph            : bolt://user-konrad.jelen-kgf-neo4j2:7687 (read-only)

Semantic nbhd    : all rels except ['ABOUT', 'MENTIONED_IN', 'SIMILAR_TO', 'SAME_AS']

Bars             : H134 gain >= 0.05  H135 legit >= 0.9  H136 >= 3 recov @prec 0.5

## Data loading

Entities with Titan embeddings, semantic-neighbor adjacency (typed relations only), SAME_AS edges
(method + evidence), and source documents. The pair set is the R12 foundation; false-merge labels
come from R11 forensics.

In [3]:
driver = GraphDatabase.driver(NEO4J_URI, auth=("neo4j", "kgfoundry"),
                              notifications_min_severity="OFF")
with driver.session() as s:
    ents = s.run(
        "MATCH (e:Entity) WHERE e.embedding IS NOT NULL "
        "RETURN e.name AS name, e.embedding AS emb, e.source_documents AS docs"
    ).data()
    adj_rows = s.run(
        "MATCH (e:Entity)-[r]-(o:Entity) WHERE NOT type(r) IN $ex "
        "RETURN e.name AS name, collect(DISTINCT o.name) AS nbrs", ex=STRUCT_EXCLUDE
    ).data()
    same_as = s.run(
        "MATCH (a:Entity)-[r:SAME_AS]->(b:Entity) "
        "RETURN a.name AS a, b.name AS b, r.method AS m, r.evidence AS ev"
    ).data()
driver.close()

EMB, DOCS = {}, {}
for r in ents:
    if r["name"] in EMB:
        continue
    v = np.asarray(r["emb"], dtype=np.float32)
    EMB[r["name"]] = v / (np.linalg.norm(v) + 1e-9)
    DOCS[r["name"]] = set(r["docs"] or [])
NBR = {r["name"]: [n for n in r["nbrs"] if n in EMB] for r in adj_rows}
NBR = defaultdict(list, NBR)

pair_file = sorted(glob.glob("../reports/matching-r12-foundation-*.json"))[-1]
pairs = json.load(open(pair_file))["pairs"]
forensics = json.load(open(sorted(glob.glob("../reports/identity-forensics-r11-final-*.json"))[-1]))
labeled_fm = forensics["labeled_false_merges"]
false_edges = {frozenset((p["a"], p["b"])) for p in labeled_fm if p["a"] in EMB and p["b"] in EMB}

degs = np.array([len(NBR[n]) for n in EMB])
rprint(f"entities [yellow]{len(EMB)}[/yellow]  SAME_AS [yellow]{len(same_as)}[/yellow]  pairs [yellow]{len(pairs)}[/yellow]  "
       f"labeled false edges [yellow]{len(false_edges)}[/yellow]")
rprint(f"semantic degree: median [yellow]{np.median(degs):.0f}[/yellow]  deg0 {int((degs==0).sum())}  deg1 {int((degs==1).sum())}  deg>=2 {int((degs>=2).sum())}")

def cos(a, b):
    return float(EMB[a] @ EMB[b])

entities 2797  SAME_AS 127  pairs 252  labeled false edges 6

semantic degree: median 1  deg0 278  deg1 1426  deg>=2 1093

## H134 - neighborhood Wasserstein vs neighbor-Jaccard

For each pair, the neighborhood-Wasserstein is the EMD between the two neighbor embedding bags
(uniform weights, cost 1-cosine); the Jaccard is the discrete overlap of neighbor names. Scored on
duplicates vs siblings (both endpoints must carry >= 1 semantic neighbor). The zero-overlap subset
- Jaccard exactly 0 but both endpoints have neighbors - is where the discrete signal is blind and
the continuous version should still carry information.

In [4]:
def nbhd_wass(a, b):
    Na, Nb = NBR[a], NBR[b]
    if not Na or not Nb:
        return None
    Va = np.stack([EMB[n] for n in Na]); Vb = np.stack([EMB[n] for n in Nb])
    M = 1.0 - Va @ Vb.T
    wa = np.full(len(Na), 1.0 / len(Na)); wb = np.full(len(Nb), 1.0 / len(Nb))
    return float(ot.emd2(wa, wb, np.ascontiguousarray(M)))

def jaccard(a, b):
    Sa, Sb = set(NBR[a]), set(NBR[b])
    if not Sa or not Sb:
        return None
    u = len(Sa | Sb)
    return len(Sa & Sb) / u if u else 0.0

task = [(p["a"], p["b"], 1 if p["cls"] in DUP_CLASSES else 0)
        for p in pairs if p["cls"] in DUP_CLASSES + SIB_CLASSES
        and p["a"] in EMB and p["b"] in EMB]

ys, s_w, s_j = [], [], []
zero_ov = []  # (y, -W) where Jaccard == 0
for a, b, y in task:
    w = nbhd_wass(a, b); j = jaccard(a, b)
    if w is None or j is None:
        continue
    ys.append(y); s_w.append(-w); s_j.append(j)
    if j == 0.0:
        zero_ov.append((y, -w))
ys = np.array(ys)

n_total = len(task)
n_scored = len(ys)
auc_w = roc_auc_score(ys, s_w) if 0 < ys.sum() < len(ys) else float("nan")
auc_j = roc_auc_score(ys, s_j) if 0 < ys.sum() < len(ys) else float("nan")
gain = auc_w - auc_j

zy = np.array([y for y, _ in zero_ov]); zs = np.array([s for _, s in zero_ov])
auc_zero = roc_auc_score(zy, zs) if len(zy) and 0 < zy.sum() < len(zy) else float("nan")
zero_signal = (not np.isnan(auc_zero)) and auc_zero > 0.5

h134 = ("CONFIRMED" if (not np.isnan(gain) and gain >= BAR_H134_GAIN and zero_signal)
        else "REFUTED")

_l = "─" * 46
rprint(f"[bold cyan]H134 - neighborhood Wasserstein vs Jaccard[/bold cyan]\n[dim]{_l}[/dim]")
rprint(f"  task pairs           : [yellow]{n_total}[/yellow]  scored (both have >=1 nbr) : [yellow]{n_scored}[/yellow]  [dim]coverage {n_scored/max(n_total,1):.0%}[/dim]")
rprint(f"  neighbor Jaccard AUC : [yellow]{auc_j:.3f}[/yellow]")
rprint(f"  neighborhood-W AUC   : [yellow]{auc_w:.3f}[/yellow]   gain [yellow]{gain:+.3f}[/yellow]  (bar +{BAR_H134_GAIN})")
rprint(f"  zero-overlap subset  : [yellow]{len(zero_ov)}[/yellow] pairs  W-AUC [yellow]{auc_zero:.3f}[/yellow]  signal>{0.5}: [yellow]{zero_signal}[/yellow]")
rprint(f"  verdict : [bold]{h134}[/bold]")

H134 - neighborhood Wasserstein vs Jaccard
──────────────────────────────────────────────

task pairs           : 126  scored (both have >=1 nbr) : 106  coverage 84%

neighbor Jaccard AUC : 0.349

neighborhood-W AUC   : 0.459   gain +0.110  (bar +0.05)

zero-overlap subset  : 50 pairs  W-AUC 0.655  signal>0.5: True

verdict : CONFIRMED

## H135 - assignment-constrained resolution replay

Each SAME_AS closure (connected component of the greedy transitive union) is re-resolved as a
low-temperature entropic-OT self-assignment: cost `1 - cosine` among members, diagonal masked, and
a SAME_AS edge survives only if its endpoints are mutual argmax of the coupling - the one-to-one
mass constraint forbids a member being the same as two mutually-unlike others. Measured against the
labeled false edges (must not survive) and the legit methods (must survive), plus the P10 family.

In [5]:
# closures (connected components) over undirected SAME_AS
adj_sa = defaultdict(set)
edges_by_method = defaultdict(list)
for r in same_as:
    a, b = r["a"], r["b"]
    if a in EMB and b in EMB:
        adj_sa[a].add(b); adj_sa[b].add(a)
        edges_by_method[r["m"]].append(frozenset((a, b)))

seen, comps = set(), []
for n0 in list(adj_sa):
    if n0 in seen:
        continue
    st, comp = [n0], set()
    while st:
        x = st.pop()
        if x in seen:
            continue
        seen.add(x); comp.add(x); st.extend(adj_sa[x] - seen)
    comps.append(comp)

def cluster_assignment(members):
    "Sinkhorn self-assignment -> set of mutual-argmax surviving pairs."
    ms = list(members)
    if len(ms) == 2:
        return {frozenset(ms)}                       # size-2: trivially mutual
    V = np.stack([EMB[m] for m in ms])
    C = 1.0 - V @ V.T
    np.fill_diagonal(C, 10.0)                         # mask self-match
    w = np.full(len(ms), 1.0 / len(ms))
    P = ot.sinkhorn(w, w, np.ascontiguousarray(C), SINKHORN_ASSIGN_REG, numItermax=5000)
    arg = P.argmax(1)
    surv = set()
    for i in range(len(ms)):
        j = arg[i]
        if arg[j] == i:                              # mutual argmax
            surv.add(frozenset((ms[i], ms[j])))
    return surv

survivors = set()
member_of_cluster = {}
for c in comps:
    survivors |= cluster_assignment(c)
    for m in c:
        member_of_cluster[m] = c

# only SAME_AS edges that lie inside a closure (all of them do) are testable
all_edges = {frozenset((r["a"], r["b"])) for r in same_as if r["a"] in EMB and r["b"] in EMB}

false_survive = sum(1 for e in false_edges if e in survivors)
legit_methods = ("normalized_name", "explicit_assertion")
legit_edges = set()
for m in legit_methods:
    legit_edges |= set(edges_by_method[m])
legit_edges -= false_edges
legit_survive = sum(1 for e in legit_edges if e in survivors)
legit_rate = legit_survive / max(len(legit_edges), 1)

# P10 family: false battery edge must drop, legit variant edges must survive
p10_false = frozenset(("AirFit P10", "Transcend P10 battery"))
p10_legit = [frozenset(("AirFit P10", "AirFit P10 for AirMini")),
             frozenset(("AirFit P10", "AirFit P10 bedside starter kit"))]
p10_false_survives = p10_false in survivors
p10_legit_survive = [e in survivors for e in p10_legit]

h135 = ("CONFIRMED" if (false_survive == 0 and legit_rate >= BAR_H135_LEGIT
                        and not p10_false_survives and any(p10_legit_survive))
        else "REFUTED")

_l = "─" * 46
rprint(f"[bold cyan]H135 - assignment-constrained resolution[/bold cyan]\n[dim]{_l}[/dim]")
rprint(f"  closures [yellow]{len(comps)}[/yellow]  SAME_AS edges [yellow]{len(all_edges)}[/yellow]  surviving edges [yellow]{len(survivors)}[/yellow]")
rprint(f"  labeled false edges surviving : [yellow]{false_survive}[/yellow] / {len(false_edges)}  (bar 0)")
rprint(f"  legit-method edges surviving  : [yellow]{legit_survive}[/yellow] / {len(legit_edges)}  rate [yellow]{legit_rate:.2f}[/yellow]  (bar {BAR_H135_LEGIT})")
rprint(f"  P10 false (battery) survives  : [yellow]{p10_false_survives}[/yellow]  (want False)")
rprint(f"  P10 legit variants survive    : [yellow]{p10_legit_survive}[/yellow]  (want >=1 True)")
rprint(f"  verdict : [bold]{h135}[/bold]")

/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:630: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:642: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)


H135 - assignment-constrained resolution
──────────────────────────────────────────────

closures 46  SAME_AS edges 127  surviving edges 78

labeled false edges surviving : 1 / 6  (bar 0)

legit-method edges surviving  : 21 / 29  rate 0.72  (bar 0.9)

P10 false (battery) survives  : False  (want False)

P10 legit variants survive    : [False, True]  (want >=1 True)

verdict : REFUTED

## H136 - Gromov-Wasserstein structural twins

For each candidate pair, the 1-hop induced subgraph (entity + semantic neighbors) becomes a metric
space via intra-cosine distances; GW distance aligns the two spaces without a shared frame. The
target is duplicates that BOTH cosine (< 0.85) and string (Jaro-Winkler < 0.70) rank below
threshold - the code-vs-name pairs. GW-similarity ranks all candidates; the flagged head is
inspected for unique recoveries. Subgraphs of size 1 make GW degenerate - coverage is reported.

In [6]:
def subgraph_C(name):
    "Intra-cosine distance matrix of {name} + its semantic neighbors."
    nodes = [name] + NBR[name]
    V = np.stack([EMB[n] for n in nodes])
    return 1.0 - V @ V.T, len(nodes)

def gw_dist(a, b):
    Ca, na = subgraph_C(a); Cb, nb = subgraph_C(b)
    pa = np.full(na, 1.0 / na); pb = np.full(nb, 1.0 / nb)
    try:
        g = ot.gromov.gromov_wasserstein2(np.ascontiguousarray(Ca), np.ascontiguousarray(Cb),
                                          pa, pb, loss_fun="square_loss")
        return float(g), na, nb
    except Exception:
        return None, na, nb

# candidate pairs = all labeled pairs; target = duplicates missed by cosine AND string
cand = [(p["a"], p["b"], 1 if p["cls"] in DUP_CLASSES else 0)
        for p in pairs if p["a"] in EMB and p["b"] in EMB
        and p["cls"] in DUP_CLASSES + SIB_CLASSES + ("random",)]

rows = []
degenerate = 0
for a, b, y in cand:
    g, na, nb = gw_dist(a, b)
    if g is None:
        continue
    is_dup = y == 1
    missed = is_dup and cos(a, b) < COS_MISS and jw(a, b) < JW_MISS
    if na < 2 or nb < 2:
        degenerate += 1
    rows.append({"a": a, "b": b, "y": y, "gw": g, "sim": -g,
                 "na": na, "nb": nb, "cos": cos(a, b), "jw": jw(a, b),
                 "missed_dup": int(missed)})

n_missed = sum(r["missed_dup"] for r in rows)
# flagged head = top-K by GW similarity; K = number of missed duplicates targeted (min 10)
K = max(10, n_missed)
head = sorted(rows, key=lambda r: -r["sim"])[:K]
recov = [r for r in head if r["missed_dup"]]
# precision in head = fraction of head that are true duplicates (any dup, not only missed)
head_prec = sum(r["y"] for r in head) / len(head) if head else 0.0
n_recov = len(recov)
non_degen = sum(1 for r in rows if r["na"] >= 2 and r["nb"] >= 2)

h136 = ("CONFIRMED" if (n_recov >= BAR_H136_RECOV and head_prec >= BAR_H136_PREC)
        else "REFUTED")

_l = "─" * 46
rprint(f"[bold cyan]H136 - Gromov-Wasserstein twins[/bold cyan]\n[dim]{_l}[/dim]")
rprint(f"  candidate pairs      : [yellow]{len(rows)}[/yellow]  non-degenerate subgraphs (both >=2 nodes) : [yellow]{non_degen}[/yellow]  degenerate : [yellow]{degenerate}[/yellow]")
rprint(f"  duplicates missed by cos<{COS_MISS} & JW<{JW_MISS} : [yellow]{n_missed}[/yellow] (the recovery target)")
rprint(f"  flagged head K={K}    : recoveries [yellow]{n_recov}[/yellow] (bar {BAR_H136_RECOV})  head precision [yellow]{head_prec:.2f}[/yellow] (bar {BAR_H136_PREC})")
rprint(f"  verdict : [bold]{h136}[/bold]")
for r in recov[:8]:
    rprint(f"  [green]recovered[/green] gw={r['gw']:.3f} cos={r['cos']:.2f} jw={r['jw']:.2f}  [dim]{r['a'][:28]} || {r['b'][:28]}[/dim]")

H136 - Gromov-Wasserstein twins
──────────────────────────────────────────────

candidate pairs      : 246  non-degenerate subgraphs (both >=2 nodes) : 203  degenerate : 43

duplicates missed by cos<0.85 & JW<0.7 : 0 (the recovery target)

flagged head K=10    : recoveries 0 (bar 3)  head precision 0.70 (bar 0.5)

verdict : REFUTED

## Report

In [7]:
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = Path("../reports") / f"transport-r13-structure-{stamp}.json"
report = {
    "pair_file": Path(pair_file).name,
    "semantic_degree": {"median": float(np.median(degs)), "deg0": int((degs == 0).sum()),
                        "deg1": int((degs == 1).sum()), "deg_ge2": int((degs >= 2).sum())},
    "h134": {"n_task": n_total, "n_scored": n_scored, "coverage": n_scored / max(n_total, 1),
             "auc_jaccard": None if np.isnan(auc_j) else float(auc_j),
             "auc_nbhd_wass": None if np.isnan(auc_w) else float(auc_w), "gain": float(gain),
             "n_zero_overlap": len(zero_ov), "auc_zero_overlap": None if np.isnan(auc_zero) else float(auc_zero),
             "zero_signal": bool(zero_signal), "bar": BAR_H134_GAIN, "verdict": h134},
    "h135": {"n_closures": len(comps), "n_edges": len(all_edges), "n_survivors": len(survivors),
             "false_survive": int(false_survive), "n_false": len(false_edges),
             "legit_survive": int(legit_survive), "n_legit": len(legit_edges), "legit_rate": float(legit_rate),
             "p10_false_survives": bool(p10_false_survives), "p10_legit_survive": [bool(x) for x in p10_legit_survive],
             "bar_legit": BAR_H135_LEGIT, "verdict": h135},
    "h136": {"n_candidates": len(rows), "non_degenerate": non_degen, "degenerate": degenerate,
             "n_missed_target": n_missed, "K": K, "n_recoveries": n_recov, "head_precision": float(head_prec),
             "bar_recov": BAR_H136_RECOV, "bar_prec": BAR_H136_PREC, "verdict": h136,
             "recoveries": [{"a": r["a"], "b": r["b"], "gw": r["gw"], "cos": r["cos"], "jw": r["jw"]} for r in recov]},
}
out.write_text(json.dumps(report, indent=2, default=str))
rprint("saved", str(out))
rprint(f"[bold]VERDICTS[/bold]  H134 {h134}  H135 {h135}  H136 {h136}")

saved ../reports/transport-r13-structure-20260707-091503.json

VERDICTS  H134 CONFIRMED  H135 REFUTED  H136 REFUTED